# Phase 4 — seal-ocr-rec-v2

Before running, upload `recognition_dataset_v2.zip` to `MyDrive/seal-ocr-phase4/`, then choose a GPU runtime in Colab. If you previously experimented with package installs, select **Runtime → Disconnect and delete runtime** first, then run this notebook once from top to bottom. This notebook never reads the frozen `benchmark/` dataset.


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, json, os, re, shutil, subprocess, sys, zipfile
from datetime import datetime, timezone

MODEL_VERSION = 'seal-ocr-rec-v2'
SEED = 20260918
RESUME = False  # Set True only to continue the saved latest checkpoint.
DRIVE_ROOT = Path('/content/drive/MyDrive/seal-ocr-phase4')
DATASET_ZIP = DRIVE_ROOT / 'recognition_dataset_v2.zip'
RUN_DIR = DRIVE_ROOT / 'runs' / MODEL_VERSION
WORK = Path('/content/seal-ocr-phase4')

drive.mount('/content/drive')
assert DATASET_ZIP.is_file(), f'Upload {DATASET_ZIP} first'
if RUN_DIR.exists() and not RESUME:
    raise FileExistsError(f'{RUN_DIR} already exists; set RESUME=True to continue it')
RUN_DIR.mkdir(parents=True, exist_ok=True)
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir()

def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

DATASET_ZIP_SHA256 = sha256(DATASET_ZIP)
print({'runDir': str(RUN_DIR), 'datasetZipSha256': DATASET_ZIP_SHA256})


In [ ]:
# Safely unpack and validate the exact Phase 3 artifact before any model download.
with zipfile.ZipFile(DATASET_ZIP) as archive:
    names = archive.namelist()
    if any(Path(name).is_absolute() or '..' in Path(name).parts for name in names):
        raise ValueError('Unsafe path inside recognition_dataset_v2.zip')
    archive.extractall(WORK)
DATASET = WORK
metadata = json.loads((DATASET / 'metadata.json').read_text())
expected = {
    'datasetVersion': 'seal-recognition-v2',
    'sourceDatasetSha256': 'c6ed49813f1145adea6edc3e2de6d07c76b2f144af238056bdc97c7b832a07e2',
    'datasetContentSha256': '7fab45cc90fb5f5243c91047a95f791b00e3b1e21bdf35eb61107751e3474f1a',
    'sourceSamples': 300, 'acceptedSamples': 291, 'trainSamples': 4500,
    'validationSamples': 500, 'seed': 20260917,
}
for key, value in expected.items():
    if key == 'datasetContentSha256':
        continue
    if metadata.get(key) != value:
        raise ValueError(f'Dataset metadata mismatch for {key}: {metadata.get(key)!r}')

from PIL import Image
def read_labels(name):
    rows = {}
    for number, line in enumerate((DATASET / name).read_text().splitlines(), 1):
        try:
            crop, truth = line.split('\t', 1)
        except ValueError as exc:
            raise ValueError(f'Malformed {name}:{number}') from exc
        path = (DATASET / crop).resolve()
        if Path(crop).is_absolute() or not path.is_relative_to(DATASET) or not path.is_file():
            raise ValueError(f'Missing or unsafe crop: {crop}')
        if crop in rows or not re.fullmatch(r'[A-Z0-9]{5,20}', truth):
            raise ValueError(f'Invalid label: {name}:{number}')
        with Image.open(path) as image:
            image.verify()
        rows[crop] = truth
    return rows

def content_sha256(root):
    digest = hashlib.sha256()
    for path in sorted((item for item in root.rglob('*') if item.is_file()), key=lambda item: item.relative_to(root).as_posix()):
        digest.update(path.relative_to(root).as_posix().encode() + b'\0')
        with path.open('rb') as source:
            for chunk in iter(lambda: source.read(1024 * 1024), b''):
                digest.update(chunk)
    return digest.hexdigest()

train, validation = read_labels('train.txt'), read_labels('val.txt')
assert not set(train) & set(validation)
assert len(train) == 4500 and len(validation) == 500
assert content_sha256(DATASET) == expected['datasetContentSha256']
print({'dataset': str(DATASET), 'train': len(train), 'validation': len(validation)})


In [ ]:
# Pin PaddleOCR source and the GPU wheel. CUDA 12.6 wheel bundles its CUDA dependencies.
PADDLE = Path('/content/PaddleOCR')
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'v3.7.0',
                'https://github.com/PaddlePaddle/PaddleOCR.git', str(PADDLE)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'paddlepaddle'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall',
                'paddlepaddle-gpu==3.2.0', '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cu126/'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PADDLE / 'requirements.txt'), 'pyyaml'], check=True)
# Training uses the cloned ppocr source directly. Avoid the high-level paddleocr package,
# which imports PaddleX and Colab's PyTorch build after Paddle changes NCCL.
import paddle
assert paddle.device.is_compiled_with_cuda(), 'Paddle GPU wheel is not active'
paddle.utils.run_check()
PADDLEOCR_VERSION = '3.7.0'
PADDLE_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PADDLE, text=True).strip()
print({'paddle': paddle.__version__, 'paddleocr': PADDLEOCR_VERSION,
       'device': paddle.device.get_device(), 'paddleocrCommit': PADDLE_COMMIT})


In [ ]:
# Derive the actual config from the official v3.7.0 Medium config.
import urllib.request, yaml
CONFIG_SOURCE = 'https://raw.githubusercontent.com/PaddlePaddle/PaddleOCR/v3.7.0/configs/rec/PP-OCRv6/PP-OCRv6_medium_rec.yml'
source_config = WORK / 'official_PP-OCRv6_medium_rec.yml'
urllib.request.urlretrieve(CONFIG_SOURCE, source_config)
config = yaml.safe_load(source_config.read_text())
config['Global'].update({'epoch_num': 60, 'seed': SEED, 'save_model_dir': str(RUN_DIR / 'checkpoints'),
    'save_epoch_step': 50, 'eval_batch_step': [0, 281], 'distributed': False,
    'use_visualdl': False, 'max_text_length': 25})
config['Optimizer']['lr'].update({'learning_rate': 0.00005, 'warmup_epoch': 2})
train_config = config['Train']
train_config['dataset']['data_dir'] = str(DATASET)
train_config['dataset']['label_file_list'] = [str(DATASET / 'train.txt')]
train_config['dataset']['ratio_list'] = [1.0]
train_config['dataset']['transforms'] = [
    {'DecodeImage': {'img_mode': 'BGR', 'channel_first': False}},
    {'RecAug': {'tia_prob': 0.10, 'crop_prob': 0.0, 'reverse_prob': 0.0, 'noise_prob': 0.15,
                'jitter_prob': 0.05, 'blur_prob': 0.10, 'hsv_aug_prob': 0.15}},
    {'MultiLabelEncode': {'gtc_encode': 'NRTRLabelEncode'}},
    {'KeepKeys': {'keep_keys': ['image', 'label_ctc', 'label_gtc', 'length', 'valid_ratio']}},
]
train_config['sampler']['first_bs'] = 16
train_config['loader'].update({'batch_size_per_card': 16, 'drop_last': False, 'num_workers': 2})
config['Eval']['dataset']['data_dir'] = str(DATASET)
config['Eval']['dataset']['label_file_list'] = [str(DATASET / 'val.txt')]
config['Eval']['loader'].update({'batch_size_per_card': 64, 'num_workers': 2})
CONFIG = RUN_DIR / 'config.yml'
CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
CONFIG_SHA256 = sha256(CONFIG)
PRETRAIN = RUN_DIR / 'PP-OCRv6_medium_rec_pretrained.pdparams'
if not PRETRAIN.exists():
    urllib.request.urlretrieve('https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv6_medium_rec_pretrained.pdparams', PRETRAIN)
PRETRAIN_SHA256 = sha256(PRETRAIN)

def run_tool(name, *args):
    completed = subprocess.run([sys.executable, *args], cwd=PADDLE, text=True, capture_output=True)
    (RUN_DIR / name).write_text(completed.stdout + '\n' + completed.stderr)
    if completed.returncode:
        raise RuntimeError(f'{name} failed; inspect {RUN_DIR / name}')
    return completed.stdout + completed.stderr

# One-batch/evaluation smoke against the pretrained checkpoint before training.
run_tool('base_validation.log', 'tools/eval.py', '-c', str(CONFIG), '-o', f'Global.pretrained_model={PRETRAIN}')
BASE_INFERENCE = RUN_DIR / 'base_inference'
run_tool('base_export.log', 'tools/export_model.py', '-c', str(CONFIG), '-o',
         f'Global.pretrained_model={PRETRAIN}', f'Global.save_inference_dir={BASE_INFERENCE}')


In [ ]:
# Evaluate exported inference models without importing PaddleX or PyTorch.
import ast, csv
def distance(left, right):
    previous = list(range(len(right) + 1))
    for index, char_left in enumerate(left, 1):
        current = [index]
        for column, char_right in enumerate(right, 1):
            current.append(min(current[-1] + 1, previous[column] + 1, previous[column - 1] + (char_left != char_right)))
        previous = current
    return previous[-1]

def normalise(text):
    return ''.join(char for char in text.upper() if char.isalnum())

VALIDATION_IMAGES = WORK / 'validation_images'
VALIDATION_IMAGES.mkdir(exist_ok=True)
validation_by_name = {}
for crop, truth in validation.items():
    name = Path(crop).name
    if name in validation_by_name:
        raise ValueError(f'Duplicate validation filename: {name}')
    validation_by_name[name] = (crop, truth)
    shutil.copy2(DATASET / crop, VALIDATION_IMAGES / name)

def evaluate_inference(model_dir, stem):
    completed = subprocess.run([sys.executable, 'tools/infer/predict_rec.py',
        '--image_dir', str(VALIDATION_IMAGES), '--rec_model_dir', str(model_dir),
        '--rec_algorithm', 'SVTR_LCNet', '--rec_image_shape', '3,48,320',
        '--rec_char_dict_path', str(PADDLE / 'ppocr/utils/dict/ppocrv6_dict.txt'),
        '--rec_batch_num', '29', '--use_gpu', 'True'],
        cwd=PADDLE, text=True, capture_output=True)
    log = completed.stdout + '\n' + completed.stderr
    (RUN_DIR / f'{stem}_inference.log').write_text(log)
    if completed.returncode:
        raise RuntimeError(f'{stem} inference failed; inspect {RUN_DIR / f"{stem}_inference.log"}')
    predictions = {}
    for line in log.splitlines():
        marker = 'Predicts of '
        if marker not in line:
            continue
        image_path, result = line.split(marker, 1)[1].rsplit(':', 1)
        prediction, confidence = ast.literal_eval(result.strip())
        predictions[Path(image_path).name] = (normalise(prediction), float(confidence))
    if set(predictions) != set(validation_by_name):
        raise RuntimeError(f'Expected 29 predictions, got {len(predictions)}')
    rows = []
    for name, (crop, truth) in sorted(validation_by_name.items()):
        prediction, confidence = predictions[name]
        rows.append({'crop': crop, 'ground_truth': truth, 'prediction': prediction, 'confidence': confidence, 'exact_match': prediction == truth})
    with (RUN_DIR / f'{stem}_validation.csv').open('w', newline='') as output:
        writer = csv.DictWriter(output, fieldnames=rows[0].keys()); writer.writeheader(); writer.writerows(rows)
    metrics = {'samples': len(rows), 'exactMatches': sum(row['exact_match'] for row in rows),
               'exactMatchAccuracy': round(sum(row['exact_match'] for row in rows) / len(rows), 4),
               'cer': round(sum(distance(row['ground_truth'], row['prediction']) for row in rows) / sum(len(row['ground_truth']) for row in rows), 4)}
    (RUN_DIR / f'{stem}_validation.json').write_text(json.dumps(metrics, indent=2) + '\n')
    return metrics

base_metrics = evaluate_inference(BASE_INFERENCE, 'base')
print(base_metrics)


In [ ]:
# Fine-tune from pretrained weights. On a fresh run only, retry smaller batches after a CUDA OOM.
CHECKPOINTS = RUN_DIR / 'checkpoints'
BATCHES = [16] if RESUME else [16, 8, 4]
last_error = None
for batch in BATCHES:
    overrides = [f'Global.save_model_dir={CHECKPOINTS}', f'Train.sampler.first_bs={batch}', f'Train.loader.batch_size_per_card={batch}', 'Global.use_amp=True', 'Global.scale_loss=1024.0', 'Global.use_dynamic_loss_scaling=True']
    if RESUME:
        latest = CHECKPOINTS / 'latest'
        if not (CHECKPOINTS / 'latest.pdparams').is_file():
            raise FileNotFoundError('RESUME=True requires checkpoints/latest.pdparams in Google Drive')
        overrides.append(f'Global.checkpoints={latest}')
    else:
        overrides.append(f'Global.pretrained_model={PRETRAIN}')
    completed = subprocess.run([sys.executable, 'tools/train.py', '-c', str(CONFIG), '-o', *overrides], cwd=PADDLE, text=True, capture_output=True)
    log = completed.stdout + '\n' + completed.stderr
    (RUN_DIR / f'train_batch{batch}.log').write_text(log)
    if completed.returncode == 0:
        TRAIN_BATCH = batch
        break
    last_error = RuntimeError(f'train_batch{batch}.log failed')
    if RESUME or 'out of memory' not in log.lower() or batch == BATCHES[-1]:
        raise last_error
    shutil.rmtree(CHECKPOINTS, ignore_errors=True)
else:
    raise last_error

BEST = CHECKPOINTS / 'best_accuracy'
if not (CHECKPOINTS / 'best_accuracy.pdparams').is_file() or not (CHECKPOINTS / 'latest.pdparams').is_file():
    raise FileNotFoundError('Training did not produce best_accuracy and latest checkpoints')
run_tool('fine_tuned_validation.log', 'tools/eval.py', '-c', str(CONFIG), '-o', f'Global.pretrained_model={BEST}')
INFERENCE = RUN_DIR / 'inference'
run_tool('export.log', 'tools/export_model.py', '-c', str(CONFIG), '-o', f'Global.pretrained_model={BEST}', f'Global.save_inference_dir={INFERENCE}')
for name in ('inference.json', 'inference.pdiparams', 'inference.yml'):
    if not (INFERENCE / name).is_file():
        raise FileNotFoundError(f'Export missing {name}')
fine_tuned_metrics = evaluate_inference(INFERENCE, 'fine_tuned')

model_metadata = {'modelVersion': MODEL_VERSION, 'baseModel': 'PP-OCRv6_medium_rec', 'datasetVersion': metadata['datasetVersion'], 'datasetSha256': metadata['sourceDatasetSha256'], 'trainingDateUtc': datetime.now(timezone.utc).isoformat(timespec='seconds'), 'paddleocrTag': 'v3.7.0', 'paddleocrCommit': PADDLE_COMMIT, 'paddleVersion': paddle.__version__, 'paddleocrVersion': PADDLEOCR_VERSION, 'gpu': subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True).strip(), 'seed': SEED, 'batchSize': TRAIN_BATCH, 'configSha256': CONFIG_SHA256, 'pretrainedCheckpointSha256': PRETRAIN_SHA256, 'baseValidationMetrics': base_metrics, 'validationMetrics': fine_tuned_metrics}
(RUN_DIR / 'metadata.json').write_text(json.dumps(model_metadata, indent=2) + '\n')
MODEL_ZIP = shutil.make_archive(str(RUN_DIR / MODEL_VERSION), 'zip', root_dir=INFERENCE)
model_metadata['modelZipSha256'] = sha256(Path(MODEL_ZIP))
(RUN_DIR / 'metadata.json').write_text(json.dumps(model_metadata, indent=2) + '\n')
print({'modelZip': MODEL_ZIP, 'base': base_metrics, 'fineTuned': fine_tuned_metrics})
